## P2 - ICL with factuality incorrect data
In this notebook, we adapt a **MODEL HERE** for translating English to Swahili using in-context learning (ICL). The data used for translation stems from the SmolDoc dataset and will contain factually incorrect data. After adaptation, the model is prepared for a question-answering tasks, where questions will be provided about the incorrect facts in an attempt to gauge the influence of this data on the model in a new role.

For documentation purposes, we start off with a bit of data preprocessing to highlight certain choices, such as choosing the Swahili subset and appending annotation notes to be used by a downstream LLM.

### Questions
The questions and ground truth answers are generated by an LLM with access to notes about the document from 3 distinct annotators explaining why and how the document is factually incorrect.

### Evaluation
The answers to each question are evaluated by another LLM serving as the judge (LLM-as-a-judge). The LLM uses a **METRIC HERE** evaluation metric, where...

### Results


### Dataset exploration

In [ ]:
from utils import list_smoldoc_configs

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
from utils import get_smoldoc_dataset

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="data/smoldoc_datasets",
    force_download=False
)
datasets_dict

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

row_counts = {cfg: len(ds) for cfg, ds in datasets_dict.items()}

# Build DataFrame
df_counts = (
    pd.DataFrame(list(row_counts.items()), columns=["config", "num_topics"])
    .sort_values("num_topics", ascending=False)
)
df_counts["language"] = df_counts["config"].str.extract(r"smoldoc__([a-z]{2})")

# --- Plot setup: configs on X-axis, topics on Y-axis ---
plt.figure(figsize=(18, 8))  # wide to fit labels

bars = plt.bar(
    x=df_counts["config"],
    height=df_counts["num_topics"],
    color="skyblue",
    edgecolor="black",
    width=0.8
)

plt.ylabel("Number of rows", fontsize=12)
plt.xlabel("SmolDoc Config", fontsize=12)
plt.title("Number of rows per SmolDoc Config", fontsize=14, fontweight="bold")

# Rotate and space out config labels
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.subplots_adjust(bottom=0.35)  # space for long config names

# Add value labels rotated vertically
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2 + 0.2,
        height + 2,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=45
    )

plt.tight_layout()
plt.show()

### Extend datasets with annotation notes

In [6]:
from utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = "smoldoc__en_sw"  # We use Swahili since this config has the full 584 document translations
annotated_dataset = datasets[example_cfg]
annotated_dataset

📂 Found existing SmolDoc DatasetDict at data/smoldoc_datasets, loading from disk...
📂 Loaded DatasetDict from data/smoldoc_datasets with 102 configs.
Using cached file: data\smoldoc-factuality-ratings.json


Dataset({
    features: ['id', 'sl', 'tl', 'srcs', 'trgs', 'factuality', 'is_src_orig', 'annotator_1_label', 'annotator_1_notes', 'annotator_2_label', 'annotator_2_notes', 'annotator_3_label', 'annotator_3_notes'],
    num_rows: 584
})

### Generate questions for factuality test

In [7]:
import pandas as pd

df = pd.DataFrame(annotated_dataset)
incorrect_data = df[df["factuality"] == "has_errors"]
incorrect_data

,id,sl,tl,srcs,trgs,factuality,is_src_orig,annotator_1_label,annotator_1_notes,annotator_2_label,annotator_2_notes,annotator_3_label,annotator_3_notes
4,ethiopia_challenges__btithihhtt,en,sw,"[But in 1974, a military junta known as the De...","[Lakini mnamo 1974, kikosi cha wanamgambo kili...",has_errors,True,Minor Issue(s),"This paragraph contains minor issue: the ""peac...",No Issues,This is a correct account of Ethiopia's histro...,No Issues,The claims made in the Paragraph about the Eth...
6,topic_260__mtaftfttit,en,sw,[Movies have long been a powerful force in sha...,[Filamu zimekuwa na ushawishi mkubwa mno katik...,has_errors,True,Minor Issue(s),"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Not Sure,The claims are generally true. But I could not...
13,topic_493__isitgsgshgsg,en,sw,[Indira Gandhi was the first and only woman to...,[Indira Gandhi alikuwa mwanamke wa kwanza na w...,has_errors,True,Minor Issue(s),There are minor inaccuracies in this paragraph...,Minor Issue(s),Indira became member of the Parliament in 1964...,Minor Issue(s),Most of the biography about Indira Gandhi is a...
23,topic_131__gtttgtiigt,en,sw,[Grace: Can you tell me a little bit about its...,[Grace: Unaweza kunielezea kidogo kuhusu histo...,has_errors,True,Minor Issue(s),Tea was actually introduced in North America i...,Minor Issue(s),Tea was actually introduced in North America i...,Minor Issue(s),The history of tea in China is accurate. Howev...
33,custom_4__iifdfdfd,en,sw,"[I'm Dr. Boakye, a pediatrician at the Korle B...","[Mimi ni Dkt. Boakye, daktari wa watoto katika...",has_errors,True,Minor Issue(s),The cases reported are more than the real ones...,Clear Issue(s),There were approximately 3.5 million reported ...,Clear Issue(s),"The claim that in 2020, there were over 24 mil..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,topic_41__iotot,en,sw,"[In the moments after a great battle, when the...","[Muda baada ya mapigano makubwa, wakati ambapo...",has_errors,True,Minor Issue(s),"“In Flanders Fields” is a real poem, but it wa...",Minor Issue(s),"While the poppy is a symbol of remembrance, th...",Minor Issue(s),"The poem ""In Flanders Fields"" was actually wri..."
566,kenya_historical__tittti,en,sw,[The Mau Mau Uprising was a major event in Ken...,[Machafuko ya Mau Mau yalikuwa tukio kuu katik...,has_errors,True,Minor Issue(s),While the text provides an overall accurate su...,Minor Issue(s),The uprising did include people outside of Kik...,No Issues,Claims are accurate.
575,topic_147__iiithirotatt,en,sw,"[""Insomnia is a common sleep disorder that can...","[""Kukosa usingizi ni tatizo la kawaida la kula...",has_errors,True,Minor Issue(s),"The factual claims about insomnia, its causes,...",Minor Issue(s),Commercial activity isn't something that has c...,Minor Issue(s),"This claim cannot be verified ""The findings of..."
579,topic_488__tiiithiontbtantbtf,en,sw,[The newspaper is a vital part of any communit...,[Gazeti ni sehemu muhimu katika jumuiya yoyote...,has_errors,True,Minor Issue(s),The claims are broadly accurate but could bene...,No Issues,Correct generalized statements regarding newsp...,Minor Issue(s),"The factuality of the information ""Blogs and w..."


In [8]:
QUESTION_GEN_SYSTEM_PROMPT = """
You are an expert dataset creator for factuality evaluation.
Your task is to read an English document (from the SmolDoc dataset) and produce a small set of factual question, answer pairs that test a model's factual understanding of the text.

Your goals:
1. Create one or more (up to three) concise, factual, and self-contained questions based on the given document. Only make one question per issue. Only if an issue from an annotator is semantically different to another annotator's issue, make an extra question about it.
2. Each question must have one short, unambiguous gold answer that is explicitly supported by the text.
3. Questions should be neither trivial nor adversarial — they should test meaningful factual comprehension, not obscure details or wordplay.
4. Do not explicitly refer to the source document, annotators or the issues in your question. The recipient of the question will only see the source document and then be asked a question about it. Refer only to the CONTENT of the document and base your question on the issues specified by the annotators.

Output only valid JSON, following this structure:

[
  {
    "question": "<English question>",
    "answer": "<short correct English answer>"
  }
]

- Limit each question to less than 25 words.
- Limit each answer to less than 10 words.
"""

In [9]:
from pipeline import parse_response
from llm_chat import CachedLLMChat, LLMChat, OpenAIChatter
from tqdm.notebook import tqdm

chatter = OpenAIChatter()
chat = LLMChat(chatter)
chat = CachedLLMChat(chat, cache_file_path="data/factuality_question_gen_cache.pkl")

questions_with_answers: list[dict[str, str]] = []

for idx, row in tqdm(incorrect_data.iterrows(), total=len(incorrect_data), desc="Generating questions"):
    id = row["id"]
    srcs = " ".join(row["srcs"])
    errors = "\n\n".join(
        (row["annotator_1_notes"], row["annotator_2_notes"], row["annotator_3_notes"])
    )
    chat.add_message("system", QUESTION_GEN_SYSTEM_PROMPT)
    response, thoughts = chat.chat(
        f"""Here is the source document: {srcs}\n
        Annotators have noted the following issues:
        {errors}\n
        Generate question and answers in the specified JSON format that adheres to the goals and limitations given."""
    )
    parsed_json = parse_response(response, id)
    questions_dicts = parsed_json or []
    for dict in questions_dicts: # Add debugging information
        dict["reasoning"] = thoughts
        dict["source_document"] = row["srcs"]
        dict["annotator_1_notes"] = row["annotator_1_notes"]
        dict["annotator_2_notes"] = row["annotator_2_notes"]
        dict["annotator_3_notes"] = row["annotator_3_notes"]

    questions_with_answers.extend(questions_dicts)
    chat.reset()

Generating questions:   0%|          | 0/102 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
df_questions = pd.DataFrame(questions_with_answers)
df_questions = df_questions[["topic_id", "question", "answer", "source_document", "annotator_1_notes", "annotator_2_notes", "annotator_3_notes", "reasoning"]] # Change the order of columns
df_questions